# 08 · Design and architecture

**Day 2 · about 60 minutes live**

Day 1 gave you the building blocks: classes, inheritance, abstraction, polymorphism. This notebook is about
**arranging** them. A pile of correct classes can still be hard to change, hard to test and hard to explain.
Software architecture is the set of decisions about *which class is responsible for what* and *who is
allowed to depend on whom*.

We start from a class that does everything, find out why that hurts, and take it apart with four ideas:
single responsibility, composition over inheritance, layers, and dependency injection. Along the way you
learn to read and sketch UML class diagrams.

**Learning goals.** By the end you can ...

- name the responsibilities of a class and spot a class that has too many `[Analyse, level 2]`
- explain the difference between "is-a" (inheritance) and "has-a" (composition) and choose between them `[Understand, level 2]`
- refactor a class so its dependencies are passed in, and test it with a fake data source `[Apply, level 2]`
- read and sketch a UML class diagram of a small package `[Communicate, level 2]`
- review a design and propose a better structure, explaining the trade-offs `[Design, level 3]`

**Before you start:** notebooks 04 (abstraction), 05 (duck typing and protocols) and 07 (dataclasses).

Exercises come in three levels: **Core** (everyone), **Stretch** (if you finish early), **Challenge** (for
experienced programmers). Check cells fail until you solve the exercise above them; that is expected.

In [ ]:
import pickle
import sqlite3
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PATH = Path("../data/raw/telco_churn.csv")
raw = pd.read_csv(DATA_PATH)

NUMERIC = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
CATEGORICAL = ["Contract", "InternetService", "PaymentMethod"]


def clean_customers(raw: pd.DataFrame) -> pd.DataFrame:
    """Same cleaning rules as src/churn/data.py, in compact form."""
    df = raw.copy().set_index("customerID")
    total = pd.to_numeric(df["TotalCharges"].astype(str).str.strip(), errors="coerce")
    df["TotalCharges"] = total.fillna(0.0)  # blanks are new customers (tenure 0), not yet billed
    if "Churn" in df.columns:
        df["Churn"] = (df["Churn"] == "Yes").astype(int)
    return df


print(raw.shape)

## 1. The starting point: a class that does everything

After notebook 01 a natural first step is to wrap the warm-up notebook in one class. Here it is. It works.
Read it once, top to bottom.

In [ ]:
class ChurnPipeline:
    """Loads the data, cleans it, trains a model, evaluates it and saves it."""

    def __init__(self, csv_path: str | Path, model_dir: str | Path):
        self.df = pd.read_csv(csv_path)  # reads a file as soon as the object is created
        self.model_dir = Path(model_dir)
        self.model = RandomForestClassifier(n_estimators=200, min_samples_leaf=20, random_state=42)

    def run(self) -> None:
        df = self.df.drop(columns=["customerID"])
        df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
        df["Churn"] = (df["Churn"] == "Yes").astype(int)
        df = pd.get_dummies(df, columns=["Contract", "InternetService", "PaymentMethod"])
        features = [
            c
            for c in df.columns
            if c.startswith(("Contract_", "InternetService_", "PaymentMethod_"))
        ]
        features += ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
        self.features = features

        X_train, X_test, y_train, y_test = train_test_split(
            df[features], df["Churn"], test_size=0.2, random_state=42, stratify=df["Churn"]
        )
        self.model.fit(X_train, y_train)
        print(classification_report(y_test, self.model.predict(X_test), digits=2))

        self.model_dir.mkdir(parents=True, exist_ok=True)
        with open(self.model_dir / "churn_model.pkl", "wb") as f:
            pickle.dump(self.model, f)
        print("saved to", self.model_dir / "churn_model.pkl")

    def score_new_customers(self, new_customers: pd.DataFrame) -> np.ndarray:
        df = new_customers.drop(columns=["customerID"])
        df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
        df = pd.get_dummies(df, columns=["Contract", "InternetService", "PaymentMethod"])
        return self.model.predict(df.reindex(columns=self.features, fill_value=0))


with tempfile.TemporaryDirectory() as tmp:
    pipeline = ChurnPipeline(DATA_PATH, tmp)
    pipeline.run()

**Think, then pair (3 minutes).** For each change request below, which lines of `ChurnPipeline` would you
have to touch? Could you make the change without risking the others?

1. Next quarter the customers come from the company's database instead of a CSV export.
2. You want a unit test for the cleaning rules, without the 1 MB CSV and without training a forest.
3. Marketing wants to try a logistic regression instead of the random forest.
4. The monthly scoring job must not print a classification report (there are no labels).
5. The cleaning rule for `TotalCharges` changes. Did you remember it is written twice?

<details><summary>Discussion</summary>

Almost every request touches `__init__` or `run`, because loading, cleaning, encoding, training, reporting and
saving are all tangled together. Cleaning is duplicated in `score_new_customers`, so a fix in one place leaves
the other one wrong. Testing anything requires a real CSV on disk (the constructor reads it) and a real model
fit. Nothing is wrong with any single line; the problem is the **structure**.
</details>

## 2. Single responsibility: one reason to change

A useful test for a class: **how many different people or events could force it to change?**
`ChurnPipeline` changes when the data source changes (data engineering), when cleaning rules change (domain
knowledge), when the model changes (data science), when the report format changes (stakeholders) and when
the storage changes (operations). Five reasons to change, five ways to break each other.

The **single responsibility principle** says each class should have one reason to change. For our pipeline
that suggests separate pieces:

| Responsibility | Piece | Changes when ... |
|---|---|---|
| Get raw customers from somewhere | a *loader* (CSV, database, test fixture) | the data source changes |
| Turn the raw export into correct types | a cleaning function | the export format changes |
| Turn customers into model inputs, consistently | a feature pipeline (notebook 04) | features change |
| Predict churn | a model that *has* a feature pipeline and an estimator | the modelling changes |
| Judge predictions | `ConfusionMatrix`, `RetentionCost` (notebook 03) | the business metric changes |
| Wire everything for one run | an experiment function or runner | the workflow changes |

This is not about making classes small for its own sake. A two-line class that nobody needs is worse than
a forty-line class with one clear job. The goal is that a change request lands in **one** place.

## 3. Composition over inheritance

Inheritance means **is-a**: a `OneHotEncoder` *is a* `Transformer`. Composition means **has-a**: a churn
model *has a* feature pipeline and *has an* estimator, and delegates work to them.

A tempting but poor design is to make the churn model *inherit* from a random forest:

```python
class ChurnForest(RandomForestClassifier):     # is a ChurnForest really "a random forest"?
    def fit(self, df, y): ...                   # preprocesses, then calls super().fit
```

It looks like less code, but now the churn model is welded to one algorithm. Trying logistic regression means
a second class (`ChurnLogReg(LogisticRegression)`) that duplicates the preprocessing. Every public method and
attribute of `RandomForestClassifier` (dozens of them) becomes part of your churn model's interface, and
scikit-learn's own machinery (`clone`, `get_params`) breaks because the `__init__` signature changed.

With composition, the churn model holds the pieces and stays small:

In [ ]:
class ChurnModel:
    """Preprocessing + any probabilistic classifier + a decision threshold."""

    def __init__(self, preprocessor, estimator, threshold: float = 0.5) -> None:
        self.preprocessor = preprocessor  # has-a: anything with fit_transform / transform
        self.estimator = estimator  # has-a: anything with fit / predict_proba
        self.threshold = threshold

    def fit(self, X: pd.DataFrame, y: pd.Series) -> "ChurnModel":
        self.estimator.fit(self.preprocessor.fit_transform(X), y)
        return self

    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        return self.estimator.predict_proba(self.preprocessor.transform(X))[:, 1]

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        return (self.predict_proba(X) >= self.threshold).astype(int)


def make_preprocessor() -> ColumnTransformer:
    # scikit-learn's ColumnTransformer plays the role of our FeaturePipeline from notebook 04
    return ColumnTransformer(
        [
            ("categories", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
            ("numbers", StandardScaler(), NUMERIC),
        ]
    )


customers = clean_customers(raw)
X, y = customers.drop(columns=["Churn"]), customers["Churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

estimators = {
    "logistic regression": LogisticRegression(max_iter=1000, random_state=42),
    "random forest": RandomForestClassifier(n_estimators=200, min_samples_leaf=20, random_state=42),
}
for name, estimator in estimators.items():
    model = ChurnModel(make_preprocessor(), estimator, threshold=0.3).fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(
        f"{name:20s} precision {precision_score(y_test, y_pred):.2f}  "
        f"recall {recall_score(y_test, y_pred):.2f}"
    )

Switching algorithms is now a matter of passing a different object. The preprocessing exists once, so the
training data and next month's batch are guaranteed to be encoded the same way.

**When is inheritance the right tool?** When the subclass really is a special case of the base class and
should be usable everywhere the base class is (all our transformers can go into any pipeline), and when you
own both classes. Use it to share an *interface* and a little template code, as with `Transformer` in
notebook 04. Do not use it just to reuse a few methods from an unrelated class.

### Check your understanding

For each pair, is it "is-a" (inheritance) or "has-a" (composition)?

1. `MedianImputer` and `Transformer`
2. `ExperimentResult` and `ConfusionMatrix`
3. A `MonthlyScoringJob` and a `ChurnModel`
4. `LogisticRegression` and scikit-learn's `ClassifierMixin`

<details><summary>Answer</summary>

1. is-a: an imputer is a transformer and must honour the `fit`/`transform` contract.
2. has-a: a result contains a confusion matrix (plus other numbers).
3. has-a: the job uses a model; a scoring job is not a kind of model.
4. is-a, in the special form of a **mixin**: `LogisticRegression` inherits `score()` from `ClassifierMixin`.
   Mixins are small classes designed to add one capability through inheritance (notebook 05).
</details>

## 4. Layers: who may depend on whom

Once responsibilities are separated, arrange them in **layers**. Each layer only knows about the layers
below it, never about the ones above.

```mermaid
flowchart TB
    A["Application layer<br/>CLI, scheduled job, notebook, API endpoint<br/><i>__main__.py, experiment.py</i>"]
    D["Domain layer<br/>features, model, evaluation: the data science<br/><i>features.py, model.py, evaluation.py</i>"]
    I["Data access layer<br/>loading and cleaning customers<br/><i>data.py</i>"]
    C["Shared definitions<br/><i>config.py, errors.py</i>"]
    A --> D
    A --> I
    D --> C
    I --> C
    A --> C
```

(The file names refer to the reference package in `src/churn`.)

The rule that matters is the **direction of the arrows**:

- `model.py` never imports `__main__.py` or reads command line arguments. So you can use the model from a
  notebook, a test, a batch job or a web service without changes.
- `data.py` knows nothing about models. Moving from CSV to a database touches only the data access layer.
- The application layer is the only place that knows about *everything*. It is thin: it wires the pieces
  together, configures logging, and turns errors into exit codes or messages.

You will meet many names for this idea (layered, hexagonal, "ports and adapters", clean architecture). For
data science projects the core is always the same: keep the **science** (features, model, metrics) free of
**plumbing** (files, databases, command lines, web frameworks).

## 5. Dependency injection: pass things in, do not create them inside

Look at this scoring job. It is short and seems harmless:

```python
class MonthlyScoringJob:
    def __init__(self):
        self.loader = CSVLoader("/mnt/exports/customers_2026_10.csv")   # created inside
        self.model = ChurnModel.load("/mnt/models/churn_v3.joblib")       # created inside

    def run(self) -> list[str]:
        ...
```

To test `run()` you now need that exact file and that exact model on disk. To score November you must edit
the code. The class *hides* its dependencies.

**Dependency injection** is a big name for a simple habit: an object receives the things it depends on
(through `__init__` or as function arguments) instead of creating them itself. The caller decides which
loader and which model to use.

In [ ]:
class MonthlyScoringJob:
    """Flag the customers the retention team should contact this month."""

    def __init__(self, loader, model: ChurnModel) -> None:
        self.loader = loader  # anything with a .load() method returning raw customers (duck typing)
        self.model = model

    def run(self) -> list[str]:
        batch = clean_customers(self.loader.load())
        flags = self.model.predict(batch.drop(columns=["Churn"], errors="ignore"))
        return batch.index[flags == 1].tolist()


class FixedLoader:
    """A stand-in data source for this demo: returns a DataFrame we already have."""

    def __init__(self, df: pd.DataFrame) -> None:
        self.df = df

    def load(self) -> pd.DataFrame:
        return self.df


fitted = ChurnModel(make_preprocessor(), LogisticRegression(max_iter=1000), threshold=0.3)
fitted.fit(X_train, y_train)

next_month = raw.sample(200, random_state=42).drop(columns=["Churn"])  # no labels yet
to_contact = MonthlyScoringJob(FixedLoader(next_month), fitted).run()
print(f"{len(to_contact)} of {len(next_month)} customers to contact, e.g. {to_contact[:3]}")

The same `MonthlyScoringJob` runs in production with a CSV or database loader and in a test with three
hand-written customers. Nothing in the class had to change, and the test needs no files. Because Python uses
duck typing, the loader does not even have to inherit from anything: it just needs `.load()`. In the reference
package this expectation is written down as a `Protocol` (`CustomerLoader` in `src/churn/data.py`).

### Exercise 5.1 · Core

Below is an `ExperimentRunner` that hard-codes its data source. Your task:

1. Write an `InMemoryLoader` whose `load()` returns a DataFrame given to its constructor.
2. Write `InjectedExperimentRunner`: same behaviour as `ExperimentRunner`, but it receives a loader
   (anything with `.load()` returning *raw* customers) instead of a path.
3. Run it on the first 100 raw customers with `InMemoryLoader`, and on the full CSV with `CSVLoader`.

In [ ]:
class CSVLoader:
    def __init__(self, path: str | Path) -> None:
        self.path = Path(path)

    def load(self) -> pd.DataFrame:
        return pd.read_csv(self.path)


class ExperimentRunner:
    """Trains and evaluates a churn model. The data source is hard-coded."""

    def __init__(self, path: str | Path, test_size: float = 0.2, threshold: float = 0.3) -> None:
        self.loader = CSVLoader(path)  # <- the problem
        self.test_size = test_size
        self.threshold = threshold

    def run(self) -> dict[str, float]:
        customers = clean_customers(self.loader.load())
        X, y = customers.drop(columns=["Churn"]), customers["Churn"]
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=self.test_size, random_state=42, stratify=y
        )
        model = ChurnModel(make_preprocessor(), LogisticRegression(max_iter=1000), self.threshold)
        y_pred = model.fit(X_tr, y_tr).predict(X_te)
        return {
            "precision": precision_score(y_te, y_pred, zero_division=0),
            "recall": recall_score(y_te, y_pred),
            "n_test": len(y_te),
        }


print(ExperimentRunner(DATA_PATH).run())

In [ ]:
class InMemoryLoader: ...  # Your code here


class InjectedExperimentRunner: ...  # Your code here

In [ ]:
# Check cell for Exercise 5.1
small = InjectedExperimentRunner(InMemoryLoader(raw.head(100))).run()
assert small["n_test"] == 20, small
full = InjectedExperimentRunner(CSVLoader(DATA_PATH)).run()
assert full["n_test"] == 1409, full
assert full == ExperimentRunner(DATA_PATH).run(), (
    "same data and settings should give the same result"
)
print("100 customers:", small)
print("all customers:", full)
print("All checks passed")

<details><summary>Hint</summary>

`InjectedExperimentRunner.__init__(self, loader, test_size=0.2, threshold=0.3)` stores the loader it
receives. `run()` is identical to the original. That is the whole refactoring: one line moved from inside
the class to the caller.
</details>

<details><summary>Solution</summary>

```python
class InMemoryLoader:
    """Serve a DataFrame that is already in memory (tests, notebooks, small demos)."""

    def __init__(self, raw: pd.DataFrame) -> None:
        self._raw = raw

    def load(self) -> pd.DataFrame:
        return self._raw.copy()   # a copy, so callers cannot change our data by accident


class InjectedExperimentRunner:
    """Trains and evaluates a churn model on customers from any loader."""

    def __init__(self, loader, test_size: float = 0.2, threshold: float = 0.3) -> None:
        self.loader = loader
        self.test_size = test_size
        self.threshold = threshold

    def run(self) -> dict[str, float]:
        customers = clean_customers(self.loader.load())
        X, y = customers.drop(columns=["Churn"]), customers["Churn"]
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=self.test_size, random_state=42, stratify=y
        )
        model = ChurnModel(make_preprocessor(), LogisticRegression(max_iter=1000), self.threshold)
        y_pred = model.fit(X_tr, y_tr).predict(X_te)
        return {
            "precision": precision_score(y_te, y_pred, zero_division=0),
            "recall": recall_score(y_te, y_pred),
            "n_test": len(y_te),
        }
```

The reference package does the same with a function instead of a class:
`run_experiment(config, loader=None)` in `src/churn/experiment.py` uses the CSV from the config if no loader
is given, and the tests pass an `InMemoryLoader`.
</details>

### Exercise 5.2 · Stretch

The customers move into a database. Write a `SQLiteLoader` that receives an open `sqlite3` connection and a
table name, and whose `load()` returns all rows of that table as a DataFrame (`pd.read_sql`). Then run your
`InjectedExperimentRunner` on it **without changing the runner**.

The check cell creates an in-memory database (`sqlite3.connect(":memory:")`) with 500 customers for you.

In [ ]:
class SQLiteLoader: ...  # Your code here

In [ ]:
# Check cell for Exercise 5.2
with sqlite3.connect(":memory:") as conn:
    raw.head(500).to_sql("customers", conn, index=False)
    result = InjectedExperimentRunner(SQLiteLoader(conn, "customers")).run()
assert result["n_test"] == 100, result
print(result)
print("All checks passed")

<details><summary>Solution</summary>

```python
class SQLiteLoader:
    """Load raw customers from a table in a SQLite database."""

    def __init__(self, connection: sqlite3.Connection, table: str) -> None:
        self.connection = connection
        self.table = table

    def load(self) -> pd.DataFrame:
        # The table name cannot be a query parameter in SQL. In real code, check it
        # against a list of allowed tables instead of formatting user input into SQL.
        return pd.read_sql(f"SELECT * FROM {self.table}", self.connection)
```

Note that the loader receives an *open connection* rather than a file path and credentials. Who opens and
closes the connection is the caller's business (here: the `with` block). That is dependency injection again.
</details>

## 6. UML class diagrams: drawing the design

Before refactoring a real project, sketch it. A **UML class diagram** shows classes as boxes (name,
attributes, methods) and their relationships as arrows. Nobody expects complete, formal UML in data science
teams; a quick sketch that everyone reads the same way is the goal.

We write diagrams in **Mermaid**, a text format that VS Code (with the recommended extension) and GitHub
render as pictures. Because it is text, it lives in the repo next to the code and shows up in code review.

| Mermaid | Meaning | In Python |
|---|---|---|
| `A <\|-- B` | B inherits from A (is-a) | `class B(A)` |
| `A <\|.. B` | B implements the interface A | B satisfies `Protocol` A |
| `A *-- B` | composition: A creates and owns B | `self.b = B()` in `A.__init__` |
| `A o-- B` | aggregation: A holds a B it was given | `def __init__(self, b): self.b = b` |
| `A --> B` | association: A keeps a reference to B | an attribute of type B |
| `A ..> B` | dependency: A uses B for a moment | B is a parameter or a local variable |
| `+name` / `-name` | public / internal (`_name`) | |
| `method()*` / `method()$` | abstract / static or class method | |

Here is the reference package `src/churn` as a diagram. Read it with these questions in mind:
Which classes could you replace without touching `ChurnModel`? Where is the only place that knows about
files?

```mermaid
classDiagram
    direction TB

    class ExperimentConfig {
        <<dataclass>>
        +Path data_path
        +tuple numeric_features
        +tuple categorical_features
        +str target
        +float test_size
        +int random_state
        +str model
        +dict model_params
        +float threshold
        +features() tuple
        +from_json(path)$ ExperimentConfig
    }

    class CustomerLoader {
        <<protocol>>
        +load() DataFrame
    }
    class CSVCustomerLoader {
        +Path path
        +load() DataFrame
    }
    class InMemoryLoader {
        -DataFrame _raw
        +load() DataFrame
    }

    class Transformer {
        <<abstract>>
        -bool _fitted
        +is_fitted() bool
        +fit(X) Transformer
        +transform(X) DataFrame
        +fit_transform(X) DataFrame
        -_fit(X)*
        -_transform(X)* DataFrame
    }
    class OneHotEncoder {
        +list columns
        -dict _categories
        +categories() dict
        +output_columns() list
    }
    class Standardizer {
        +list columns
        -Series _mean
        -Series _std
    }
    class FeaturePipeline {
        +list steps
        +list keep
        +output_columns() list
    }

    class ProbabilisticClassifier {
        <<protocol>>
        +fit(X, y)
        +predict_proba(X) ndarray
    }
    class ChurnModel {
        +ExperimentConfig config
        -FeaturePipeline _features
        -ProbabilisticClassifier _estimator
        +threshold float
        +fit(X, y) ChurnModel
        +predict_proba(X) ndarray
        +predict(X) ndarray
        +save(path) Path
        +load(path)$ ChurnModel
    }

    class ConfusionMatrix {
        <<dataclass>>
        +int tn
        +int fp
        +int fn
        +int tp
        +from_predictions(y_true, y_pred)$ ConfusionMatrix
        +precision() float
        +recall() float
        +f1() float
    }
    class RetentionCost {
        <<dataclass>>
        +float offer_cost
        +float customer_value
        +float acceptance_rate
        +__call__(y_true, y_pred) float
        +do_nothing(y_true) float
    }
    class ExperimentResult {
        <<dataclass>>
        +ConfusionMatrix confusion
        +float roc_auc
        +float cost
        +float cost_do_nothing
        +savings() float
    }
    class experiment {
        <<module>>
        +run_experiment(config, loader, cost)$ tuple
    }

    CustomerLoader <|.. CSVCustomerLoader
    CustomerLoader <|.. InMemoryLoader
    Transformer <|-- OneHotEncoder
    Transformer <|-- Standardizer
    Transformer <|-- FeaturePipeline
    FeaturePipeline o-- Transformer : steps
    ChurnModel *-- FeaturePipeline : builds from config
    ChurnModel o-- ProbabilisticClassifier : estimator
    ChurnModel --> ExperimentConfig
    ExperimentResult *-- ConfusionMatrix
    experiment ..> CustomerLoader : loads with
    experiment ..> ChurnModel : trains
    experiment ..> RetentionCost : evaluates with
    experiment ..> ExperimentResult : returns
```

Things to notice:

- `FeaturePipeline` both **is** a `Transformer` and **has** `Transformer`s. That combination is the
  *Composite* pattern: a pipeline of steps can be used wherever a single step can.
- `ChurnModel` *builds* its feature pipeline (composition) but *receives* its estimator if you pass one
  (aggregation). That is why the tests can plug in scikit-learn's `DummyClassifier`.
- Only the loaders know about files. Only `run_experiment` (and the CLI above it) knows about everything.
- Properties appear as methods with `()`; Mermaid has no special symbol for them.

### Exercise 6.1 · Core (pairs, then gallery walk)

Sketch the class diagram for **your** refactoring of the warm-up notebook, on paper or in the Mermaid cell
below. Use at least these names: `CustomerLoader`, `CSVCustomerLoader`, `Transformer`, `OneHotEncoder`,
`ChurnModel`, `ConfusionMatrix`, `ExperimentRunner`. For each class write 1-3 attributes and the main methods.
Choose the arrows deliberately: inheritance, composition, aggregation or dependency?

In the workshop: put your sketch on the wall or your screen, walk around, and leave one question on another
pair's diagram.

The diagram does not have to match `src/churn`. Different designs can be good; what matters is that you can
explain your arrows.

```mermaid
classDiagram
    %% Your diagram here. Example syntax:
    %% class ChurnModel {
    %%     +float threshold
    %%     +fit(X, y) ChurnModel
    %% }
    %% Transformer <|-- OneHotEncoder
    class ChurnModel
```

<details><summary>One possible solution</summary>

```mermaid
classDiagram
    class CustomerLoader {
        <<protocol>>
        +load() DataFrame
    }
    class CSVCustomerLoader {
        +Path path
        +load() DataFrame
    }
    class Transformer {
        <<abstract>>
        +fit(X)*
        +transform(X)*
        +fit_transform(X)
    }
    class OneHotEncoder {
        +list columns
        -dict _categories
    }
    class ChurnModel {
        +float threshold
        -Transformer _features
        -estimator _estimator
        +fit(X, y)
        +predict(X) ndarray
    }
    class ConfusionMatrix {
        +int tn
        +int fp
        +int fn
        +int tp
        +recall() float
    }
    class ExperimentRunner {
        -CustomerLoader _loader
        +run() ConfusionMatrix
    }
    CustomerLoader <|.. CSVCustomerLoader
    Transformer <|-- OneHotEncoder
    ChurnModel *-- OneHotEncoder
    ExperimentRunner o-- CustomerLoader : injected
    ExperimentRunner ..> ChurnModel : creates and trains
    ExperimentRunner ..> ConfusionMatrix : returns
```

Typical discussion points from the gallery walk: is the runner's loader composition or aggregation (it is
passed in, so aggregation)? Does `ChurnModel` own the encoder (yes: nobody else should fit it)? Should
`ConfusionMatrix` depend on `ChurnModel` (no: it only needs labels, which makes it reusable for any model).
</details>

## 7. The finished architecture

The reference package `src/churn` is the full version of everything in this notebook. Its layers map to
files like this:

| Layer | File | Contains |
|---|---|---|
| Application | `__main__.py` | command line, logging setup, error handling at the boundary |
| Application | `experiment.py` | `run_experiment(config, loader)`, `ExperimentResult` |
| Domain | `model.py` | `ChurnModel` (composition), `ProbabilisticClassifier` protocol |
| Domain | `features.py` | `Transformer` ABC, `OneHotEncoder`, `Standardizer`, `FeaturePipeline` |
| Domain | `evaluation.py` | `ConfusionMatrix`, `RetentionCost` |
| Data access | `data.py` | `CustomerLoader` protocol, CSV and in-memory loaders, cleaning rules |
| Shared | `config.py`, `errors.py` | `ExperimentConfig`, exception hierarchy |

Because the data source is injected, you can run the whole experiment on data that is already in memory:

In [ ]:
from churn import ExperimentConfig, InMemoryLoader, run_experiment

model, result = run_experiment(ExperimentConfig(threshold=0.3), loader=InMemoryLoader(raw))
print(model)
print(result.summary())
print()
print(result.confusion)

The same call with a different loader would train on database rows, and the tests in `tests/` use small
hand-written DataFrames. From the command line, the application layer reads the CSV path from a config file:

```bash
uv run python -m churn --config configs/baseline.json
```

You will turn the warm-up notebook into a package like this yourself in the capstone (notebook 12).

### Exercise 7.1 · Challenge: design review

A colleague asks you to review this class before it goes into the monthly job. Do not run it (it would write
a database file). Find **at least four** design problems, explain the consequence of each, and sketch (in
words or a Mermaid diagram) how you would restructure it.

```python
class ChurnReport(RandomForestClassifier):
    def __init__(self, csv_path):
        super().__init__(n_estimators=300)
        self.df = pd.read_csv(csv_path)
        self.db = sqlite3.connect("reports.db")
        self.df["TotalCharges"] = pd.to_numeric(self.df["TotalCharges"], errors="coerce")
        self.df = self.df.dropna()

    def make_report(self):
        X = pd.get_dummies(self.df.drop(columns=["customerID", "Churn"]))
        y = (self.df["Churn"] == "Yes").astype(int)
        self.fit(X, y)
        self.df["risk"] = self.predict_proba(X)[:, 1]
        top = self.df.sort_values("risk", ascending=False).head(100)
        top.to_sql("to_contact", self.db, if_exists="replace")
        html = top.to_html()
        print(html)
        return html
```

<details><summary>Review notes</summary>

1. **Inheritance used for code reuse.** A report is not a kind of random forest. The class exposes every
   forest method and parameter, and its `__init__` signature breaks scikit-learn's `get_params`/`clone`, so
   `GridSearchCV` or `cross_val_score` would fail. Use composition: the report *has* a model.
2. **Side effects in `__init__`.** Creating the object reads a CSV and opens a database file. You cannot create
   it in a test, and a failing file read happens at a surprising moment. Constructors should only store
   what they are given.
3. **Hidden, hard-coded dependencies.** The database name and the data source are fixed inside the class.
   Inject a loader and a writer (or a connection) instead.
4. **Mixed layers.** Data access (CSV, SQLite), cleaning, feature encoding, training, scoring and
   presentation (HTML, print) are all in one method. Each has a different reason to change.
5. **Data science bugs hidden by the structure:** the model is trained and evaluated on the same customers
   (the "risk" scores are in-sample and too confident); `dropna` silently removes the new customers
   (`TotalCharges` blank); `get_dummies` on each new file produces batch-dependent columns, the warm-up bug.
6. **Training inside a report.** A monthly report should *load* a trained, validated model, not retrain one
   with whatever data arrived this month.

A better structure: a loader (data access) provides cleaned customers; a `ChurnModel` trained and saved by a
separate training run (application: `run_experiment` + CLI) is loaded by a `MonthlyScoringJob` that receives
the loader, the model and a writer; a small presentation function turns the top-k table into HTML.
Each piece can be tested on its own with in-memory fakes.
</details>

## Summary

- A class that loads, cleans, trains, evaluates and saves has many reasons to change. Split it so each change
  request lands in one place (single responsibility).
- Prefer **composition** ("has-a") for assembling behaviour: a churn model *has* a feature pipeline and an
  estimator. Use **inheritance** ("is-a") for real special cases that share an interface.
- Arrange code in **layers**: data access, domain (the data science), application. Dependencies point inwards;
  the model never knows about files or command lines.
- **Dependency injection**: pass loaders, models and connections in instead of creating them inside. This makes
  code reusable and testable with in-memory fakes.
- UML class diagrams (in Mermaid, next to the code) are a quick way to discuss a design before writing it.

**Common pitfalls:** reading files or opening connections in `__init__`; inheriting from a library class to
"get its methods"; splitting code into many tiny classes without a clear responsibility each; an
"application" layer that slowly fills up with data science logic.

## Going further

- The full set of SOLID principles (single responsibility, open/closed, Liskov substitution, interface
  segregation, dependency inversion). We used S and D explicitly; the others appeared implicitly (any
  `Transformer` can replace another: that is Liskov).
- Notebook X3 shows design patterns you already use in data science libraries (Strategy, Composite, Factory,
  Adapter).
- Resources:
  - Harry Percival and Bob Gregory, *Architecture Patterns with Python* (free online): https://www.cosmicpython.com/
  - Refactoring Guru, design patterns and code smells: https://refactoring.guru/
  - Mermaid class diagram syntax: https://mermaid.js.org/syntax/classDiagram.html